# Semantic Review Resolver

This notebook processes the **unified review queue** from all semantic pipelines, applies human feedback, and updates semantic mapping tables. It's the human-in-the-loop component of the semantic layer.

## Pipeline Flow (Aligned with README_SEMANTIC.md)
- **Input**: Unified review queue from Silver layer
  - `workspace.silver.silver_semantic_review_queue` (all entity types: role, company, sector, skill)
- **Output**: 
  - Updated semantic mapping tables (`workspace.semantic.sem_*_map`)
  - Audit trail of human decisions (`workspace.gold.semantic_review_audit`)

## Features
- Process unified review queue (all entity types in one table)
- Apply human feedback and corrections
- Write approved mappings to semantic layer tables
- Track review decisions for audit and compliance
- Auto-learn patterns from approved reviews

## Architecture Note
This workflow uses the **actual implemented semantic layer architecture**:
- Review queue: Single unified table in Silver layer
- Outputs: Semantic mapping tables (sem_job_role_map, sem_company_map, sem_sector_map)
- Not the legacy Gold-layer separate queues design

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType
from datetime import datetime
import json

# Configuration
# Aligned with actual semantic layer architecture (see README_SEMANTIC.md)
CONFIG = {
    # Unified review queue (all entity types in one table)
    "review_queue": "workspace.silver.silver_semantic_review_queue",
    
    # Semantic layer output tables (canonical mappings)
    "sem_job_role_map": "workspace.semantic.sem_job_role_map",
    "sem_company_map": "workspace.semantic.sem_company_map",
    "sem_sector_map": "workspace.semantic.sem_sector_map",
    "sem_company_canonical": "workspace.semantic.sem_company_canonical",
    "sem_skill_catalog": "workspace.semantic.sem_skill_catalog",
    
    # Audit table (remains in Gold layer for operational metrics)
    "review_audit": "workspace.gold.semantic_review_audit",
    
    # Batch settings
    "batch_size": 100,
    "auto_learn_threshold": 0.8,  # Confidence threshold for auto-learning patterns
    "confidence_threshold": 0.7   # Threshold for review routing
}

print("Review Resolver Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [0]:
# Load unified review queue and split by entity type
review_queues = {}

try:
    # Load from unified Silver review queue
    unified_queue = spark.table(CONFIG["review_queue"]) \
        .filter(F.col("review_status") == "pending")
    
    total_pending = unified_queue.count()
    print(f"Loaded {total_pending} total pending reviews from {CONFIG['review_queue']}")
    
    # Split by entity_type
    for entity_type in ["role", "company", "sector", "skill"]:
        df = unified_queue.filter(F.col("entity_type") == entity_type)
        count = df.count()
        review_queues[entity_type] = df if count > 0 else None
        
        if count > 0:
            print(f"  • {entity_type}: {count} pending reviews")
            display(df.limit(5))
        else:
            print(f"  • {entity_type}: No pending reviews")
    
    print(f"\nTotal pending reviews: {total_pending}")
    
except Exception as e:
    print(f"Error loading unified review queue: {e}")
    print("Note: The unified review queue may not exist yet or may be empty.")
    for entity_type in ["role", "company", "sector", "skill"]:
        review_queues[entity_type] = None

In [0]:
# In production, this would integrate with a human review UI
# For this notebook, we'll simulate review decisions

def simulate_review_decision(row, entity_type):
    """
    Simulate human review decision.
    In production, this would come from a review UI.
    
    Returns: (action, canonical_value, confidence, notes)
    Actions: 'approve', 'reject', 'create_new', 'update_mapping'
    """
    import random
    
    # Simulate different review outcomes
    rand = random.random()
    
    if rand < 0.6:
        # Approve suggested mapping
        return ('approve', row.get('canonical_role') or row.get('canonical_company_name') or row.get('sector_name'), 
                0.95, 'Approved by reviewer')
    elif rand < 0.8:
        # Create new canonical entry
        return ('create_new', row.get('standardized_role') or row.get('extracted_company_name') or row.get('extracted_sector'),
                1.0, 'Created new canonical entry')
    elif rand < 0.9:
        # Update mapping with correction
        return ('update_mapping', f"Corrected_{entity_type}", 0.9, 'Mapping updated with correction')
    else:
        # Reject
        return ('reject', None, 0.0, 'Rejected - invalid data')

print("Review decision simulator ready")
print("In production, this would integrate with a human review UI")

In [0]:
# Process role review queue
if review_queues["role"] is not None and review_queues["role"].count() > 0:
    print("\nProcessing role reviews...")
    
    role_reviews = review_queues["role"].limit(CONFIG["batch_size"]).collect()
    
    approved_roles = []
    new_mappings = []
    rejected_roles = []
    audit_records = []
    
    for row in role_reviews:
        action, canonical_value, confidence, notes = simulate_review_decision(row.asDict(), "role")
        
        # Record audit entry
        audit_records.append((
            row['role_id'],
            'role',
            row['standardized_role'],
            canonical_value,
            action,
            confidence,
            notes,
            'system_reviewer',  # In production, this would be the actual reviewer ID
            datetime.now()
        ))
        
        if action == 'approve' and canonical_value is not None:
            approved_roles.append((
                row['role_id'],
                row['standardized_role'],
                canonical_value,
                confidence,
                'human_review',
                datetime.now()
            ))
            # Add to dictionary for future use
            new_mappings.append((
                canonical_value,
                row['standardized_role'].lower(),
                'human_review',
                confidence
            ))
        elif action == 'create_new' and canonical_value is not None:
            approved_roles.append((
                row['role_id'],
                row['standardized_role'],
                canonical_value,
                1.0,
                'new_canonical',
                datetime.now()
            ))
            new_mappings.append((
                canonical_value,
                canonical_value.lower(),
                'exact',
                1.0
            ))
    
    print(f"\nRole Review Results:")
    print(f"  Approved: {len(approved_roles)}")
    print(f"  New mappings learned: {len(new_mappings)}")
    print(f"  Rejected: {len(rejected_roles)}")
    
    # Convert to DataFrames for writing
    if approved_roles:
        roles_schema = StructType([
            StructField("role_id", StringType(), True),
            StructField("standardized_role", StringType(), True),
            StructField("canonical_role", StringType(), True),
            StructField("confidence", FloatType(), True),
            StructField("method", StringType(), True),
            StructField("processed_timestamp", TimestampType(), True)
        ])
        approved_roles_df = spark.createDataFrame(
            approved_roles,
            roles_schema
        )
        print("\nApproved roles:")
        display(approved_roles_df)
    else:
        approved_roles_df = None
    
    if new_mappings:
        mappings_schema = StructType([
            StructField("canonical_role", StringType(), True),
            StructField("alias", StringType(), True),
            StructField("match_type", StringType(), True),
            StructField("confidence_boost", FloatType(), True)
        ])
        new_role_mappings_df = spark.createDataFrame(
            new_mappings,
            mappings_schema
        )
        print("\nNew role mappings:")
        display(new_role_mappings_df)
    else:
        new_role_mappings_df = None
else:
    print("\nNo role reviews to process")
    approved_roles_df = None
    new_role_mappings_df = None
    audit_records = []

In [0]:
# Process company review queue
if review_queues["company"] is not None and review_queues["company"].count() > 0:
    print("\nProcessing company reviews...")
    
    company_reviews = review_queues["company"].limit(CONFIG["batch_size"]).collect()
    
    approved_companies = []
    new_aliases = []
    company_audit_records = []
    
    for row in company_reviews:
        row_dict = row.asDict()
        action, canonical_value, confidence, notes = simulate_review_decision(row_dict, "company")
        
        # Record audit entry
        company_audit_records.append((
            row['extraction_id'],
            'company',
            row['extracted_company_name'],
            canonical_value,
            action,
            confidence,
            notes,
            'system_reviewer',
            datetime.now()
        ))
        
        if action in ['approve', 'create_new'] and canonical_value is not None:
            company_id = row_dict.get('company_id', f"company_{hash(canonical_value) % 10**8:08d}")
            approved_companies.append((
                row['extraction_id'],
                row['extracted_company_name'],
                company_id,
                canonical_value,
                confidence,
                'human_review' if action == 'approve' else 'new_canonical',
                datetime.now()
            ))
            # Add alias
            if row['extracted_company_name'] and canonical_value and row['extracted_company_name'].lower() != canonical_value.lower():
                new_aliases.append((
                    company_id,
                    row['extracted_company_name'],
                    'human_review'
                ))
    
    print(f"\nCompany Review Results:")
    print(f"  Approved: {len(approved_companies)}")
    print(f"  New aliases learned: {len(new_aliases)}")
    
    if approved_companies:
        schema = StructType([
            StructField("extraction_id", StringType(), True),
            StructField("extracted_company_name", StringType(), True),
            StructField("company_id", StringType(), True),
            StructField("canonical_company_name", StringType(), True),
            StructField("confidence", FloatType(), True),
            StructField("method", StringType(), True),
            StructField("processed_timestamp", TimestampType(), True)
        ])
        approved_companies_df = spark.createDataFrame(
            approved_companies,
            schema
        )
        print("\nApproved companies:")
        display(approved_companies_df)
    else:
        approved_companies_df = None
    
    if new_aliases:
        alias_schema = StructType([
            StructField("company_id", StringType(), True),
            StructField("alias", StringType(), True),
            StructField("alias_type", StringType(), True)
        ])
        new_company_aliases_df = spark.createDataFrame(
            new_aliases,
            alias_schema
        )
        print("\nNew company aliases:")
        display(new_company_aliases_df)
    else:
        new_company_aliases_df = None
    
    audit_records.extend(company_audit_records)
else:
    print("\nNo company reviews to process")
    approved_companies_df = None
    new_company_aliases_df = None

In [0]:
# Process sector review queue
if review_queues["sector"] is not None and review_queues["sector"].count() > 0:
    print("\nProcessing sector reviews...")
    
    sector_reviews = review_queues["sector"].limit(CONFIG["batch_size"]).collect()
    
    approved_sectors = []
    sector_audit_records = []
    
    for row in sector_reviews:
        row_dict = row.asDict()
        action, canonical_value, confidence, notes = simulate_review_decision(row_dict, "sector")
        
        # Record audit entry
        sector_audit_records.append((
            row['extraction_id'],
            'sector',
            row['extracted_sector'],
            canonical_value,
            action,
            confidence,
            notes,
            'system_reviewer',
            datetime.now()
        ))
        
        if action in ['approve', 'create_new'] and canonical_value is not None:
            approved_sectors.append((
                row['extraction_id'],
                row['extracted_sector'],
                row_dict.get('sector_code', 'custom_001'),
                canonical_value,
                row_dict.get('taxonomy_type', 'CUSTOM'),
                confidence,
                'human_review' if action == 'approve' else 'new_canonical',
                datetime.now()
            ))
    
    print(f"\nSector Review Results:")
    print(f"  Approved: {len(approved_sectors)}")
    
    if approved_sectors:
        sectors_schema = StructType([
            StructField("extraction_id", StringType(), True),
            StructField("extracted_sector", StringType(), True),
            StructField("sector_code", StringType(), True),
            StructField("sector_name", StringType(), True),
            StructField("taxonomy_type", StringType(), True),
            StructField("confidence", FloatType(), True),
            StructField("method", StringType(), True),
            StructField("processed_timestamp", TimestampType(), True)
        ])
        approved_sectors_df = spark.createDataFrame(
            approved_sectors,
            sectors_schema
        )
        print("\nApproved sectors:")
        display(approved_sectors_df)
    else:
        approved_sectors_df = None
    
    audit_records.extend(sector_audit_records)
else:
    print("\nNo sector reviews to process")
    approved_sectors_df = None

In [0]:
# Write approved records to semantic layer mapping tables
write_count = 0

if approved_roles_df is not None:
    print(f"\nWriting {approved_roles_df.count()} approved roles to {CONFIG['sem_job_role_map']}...")
    approved_roles_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["sem_job_role_map"])
    write_count += approved_roles_df.count()
    print("✓ Role mappings written")

if approved_companies_df is not None:
    print(f"\nWriting {approved_companies_df.count()} approved companies to {CONFIG['sem_company_map']}...")
    # Write to both sem_company_map and sem_company_canonical as needed
    approved_companies_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["sem_company_map"])
    write_count += approved_companies_df.count()
    print("✓ Company mappings written")

if approved_sectors_df is not None:
    print(f"\nWriting {approved_sectors_df.count()} approved sectors to {CONFIG['sem_sector_map']}...")
    approved_sectors_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["sem_sector_map"])
    write_count += approved_sectors_df.count()
    print("✓ Sector mappings written")

print(f"\n✓ Total approved records written to semantic layer: {write_count}")

In [0]:
# Update semantic mapping tables with learned patterns
# Note: Semantic layer uses mapping tables (sem_*_map) rather than separate dictionaries
# Learned patterns are incorporated directly into the mapping tables or in-memory dictionaries

if new_role_mappings_df is not None:
    print(f"\nIncorporating {new_role_mappings_df.count()} new role mappings...")
    # Already written to sem_job_role_map in previous cell
    # In-memory dictionaries used by semantic processors can be refreshed on next run
    print("✓ Role mappings incorporated (available in sem_job_role_map)")

if new_company_aliases_df is not None:
    print(f"\nIncorporating {new_company_aliases_df.count()} new company aliases...")
    # Already written to sem_company_map in previous cell
    # Semantic company canonicalization can use these mappings on next run
    print("✓ Company aliases incorporated (available in sem_company_map)")

print("\n✓ Learned patterns incorporated into semantic mapping tables")
print("Note: Semantic processors will pick up these patterns on their next execution")

In [0]:
# Write audit records
if audit_records:
    print(f"\nWriting {len(audit_records)} audit records to {CONFIG['review_audit']}...")
    
    audit_schema = StructType([
        StructField("entity_id", StringType(), True),
        StructField("entity_type", StringType(), True),
        StructField("original_value", StringType(), True),
        StructField("canonical_value", StringType(), True),
        StructField("action", StringType(), True),
        StructField("confidence", FloatType(), True),
        StructField("notes", StringType(), True),
        StructField("reviewer_id", StringType(), True),
        StructField("review_timestamp", TimestampType(), True)
    ])
    audit_df = spark.createDataFrame(
        audit_records,
        audit_schema
    )
    
    audit_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["review_audit"])
    
    print("✓ Audit trail written")
    
    # Show audit summary
    audit_summary = audit_df.groupBy("entity_type", "action").count().orderBy("entity_type", "action")
    print("\nAudit Summary:")
    display(audit_summary)
else:
    print("\nNo audit records to write")

In [0]:
# Mark reviewed items as processed in review queues
# In production, this would update the status to 'processed' or 'completed'

print("\nUpdating review queue status...")
print("Note: In production, this would mark processed items in the review queue tables")
print("      using MERGE operations to update status from 'pending' to 'processed'")

# Example SQL for role queue update (would be executed per queue):
sample_update_sql = """
-- Example update query (not executed in this demo):
-- MERGE INTO workspace.gold.role_review_queue AS target
-- USING (SELECT role_id FROM approved_roles) AS source
-- ON target.role_id = source.role_id AND target.review_status = 'pending'
-- WHEN MATCHED THEN UPDATE SET 
--   review_status = 'processed',
--   processed_timestamp = current_timestamp()
"""

print(sample_update_sql)

In [0]:
print("\n" + "="*70)
print("SEMANTIC REVIEW RESOLVER - EXECUTION SUMMARY")
print("="*70)
print(f"Total pending reviews: {total_pending}")
print(f"Batch size processed: {CONFIG['batch_size']}")
print(f"\nApproved and written:")
if approved_roles_df:
    print(f"  Roles: {approved_roles_df.count()}")
if approved_companies_df:
    print(f"  Companies: {approved_companies_df.count()}")
if approved_sectors_df:
    print(f"  Sectors: {approved_sectors_df.count()}")
print(f"\nDictionaries updated:")
if new_role_mappings_df:
    print(f"  Role mappings: {new_role_mappings_df.count()}")
if new_company_aliases_df:
    print(f"  Company aliases: {new_company_aliases_df.count()}")
print(f"\nAudit records: {len(audit_records)}")
print("\nNote: This is a demonstration with simulated reviews.")
print("      In production, integrate with a human review UI.")
print("="*70)